# 00 — Local Smoke Test (CPU only)

**Goal:** validate the non-GPU pieces of the Counterfactual Audit Pipeline before we pay for a single second of L4 time.

**What this exercises:**
1. Library imports and version sanity checks
2. Prompt construction across the full counterfactual axis grid
3. Stratified sampling logic on a synthetic dataset (no FairFace download)
4. Statistical functions on synthetic audit predictions
5. Publication-quality static figures + interactive Plotly dashboard

**What this does NOT test:**
- Flux / PuLID / ControlNet generation (needs GPU)
- Real FairFace download (network + ~2 GB; section 3b at the bottom is optional)
- Real auditee API calls (needs credentials)

If every cell runs green, the pipeline plumbing is sound and we are safe to spend on L4.

**To run:** `pip install -e ".[dev]"` from the repo root, then open this notebook.

## 1. Imports + version sanity

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add repo src/ to path so we can import without `pip install -e .` if needed
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import cap
from cap.utils import set_global_seed

set_global_seed(42)
print(f"cap version: {cap.__version__}")
print(f"python:      {sys.version.split()[0]}")
print(f"numpy:       {np.__version__}")
print(f"pandas:      {pd.__version__}")

## 2. Prompt construction

Build the full counterfactual prompt grid for one identity and inspect a sample. Each prompt should hold pose / expression / lighting constant and only vary demographic attributes.

In [ ]:
import itertools

from cap.generator.prompts import build_demographic_prompt

fixed = {"pose": "frontal", "expression": "neutral expression", "lighting": "soft even studio lighting"}
axes = {"skin_tone": [1, 2, 3, 4, 5, 6], "gender": ["male", "female"], "age": [30]}

prompts = []
for combo in itertools.product(*axes.values()):
    av = dict(zip(axes.keys(), combo))
    prompts.append({"axis_values": av, "prompt": build_demographic_prompt(fixed, av)})

df_prompts = pd.DataFrame(prompts)
print(f"Generated {len(df_prompts)} counterfactual prompts.")
df_prompts.head(4)

In [ ]:
# Spot-check: every prompt should mention the held-constant lighting
assert df_prompts["prompt"].str.contains("soft even studio lighting").all()
# Every prompt should mention exactly one Fitzpatrick type
for prompt in df_prompts["prompt"]:
    types_mentioned = sum(f"type {n}" in prompt for n in ["I", "II", "III", "IV", "V", "VI"])
    assert types_mentioned == 1, prompt
print("✓ prompts pass structural checks")

## 3. Stratified sampling — synthetic data

We avoid downloading FairFace here. Instead we build a synthetic frame matching FairFace's schema and verify that the sampler produces the expected balanced cell counts.

In [ ]:
from cap.data.fairface import FairFaceLoader, FAIRFACE_RACE_GROUPS, FAIRFACE_GENDER_GROUPS

rng = np.random.default_rng(42)
n_synth = 5000
synth_df = pd.DataFrame({
    "race": rng.choice(FAIRFACE_RACE_GROUPS, n_synth),
    "gender": rng.choice(FAIRFACE_GENDER_GROUPS, n_synth),
    "age": rng.choice(["20-29", "30-39", "40-49", "50-59"], n_synth),
    "service_test": [False] * n_synth,
    "source_index": list(range(n_synth)),
})

loader = FairFaceLoader(output_dir="/tmp/_smoke", split="1.25")
sampled = loader.stratified_sample(synth_df, n=200, stratify_by=["race", "gender", "age"], seed=42)
print(f"Sampled {len(sampled)} rows across {sampled.groupby(['race', 'gender', 'age']).ngroups} cells")
sampled.groupby(["race", "gender", "age"]).size().describe()

Per-cell counts should be tight (target = 200 / 7 races × 2 genders × 4 ages ≈ 3.6 per cell, rounded). The min/max should not differ by more than a couple.

## 4. Statistical functions on synthetic audit data

We fabricate a plausible audit-result frame matching the real schema, then run every confirmatory test from the pre-registration. This catches schema drift and import errors in `scipy` / `statsmodels` / `pingouin`.

In [ ]:
# Synthesize 200 identities × 12 counterfactuals = 2,400 rows, 4 systems
rng = np.random.default_rng(42)
ids = [f"ff{i:06d}" for i in range(200)]
skin_tones = [1, 2, 3, 4, 5, 6]
genders = ["male", "female"]
systems = ["aws_rekognition", "azure_face", "deepface", "insightface"]

rows = []
for ident in ids:
    for s in skin_tones:
        for g in genders:
            # Inject a plausible bias pattern: error rate increases with skin tone, more for females
            base_err = 0.02 + 0.04 * (s - 1) + 0.05 * (g == "female") + 0.02 * (s - 1) * (g == "female")
            for sysname in systems:
                err = rng.random() < base_err + rng.normal(0, 0.01)
                pred = ("female" if g == "male" else "male") if err else g
                rows.append({
                    "seed_identity_id": ident, "skin_tone": s, "gender": g,
                    "auditor": sysname, "task": "gender",
                    "prediction": pred, "ground_truth": g,
                    "confidence": float(np.clip(rng.beta(8, 2) - (0.05 if err else 0), 0, 1)),
                    "error": int(err),
                })
synth_audit = pd.DataFrame(rows)
print(f"Synthesized {len(synth_audit):,} audit rows across {synth_audit['auditor'].nunique()} systems")
synth_audit.head()

In [ ]:
from cap.analysis import (
    counterfactual_flip_rate, intersectional_error_table,
    two_way_anova, mcnemars_paired, ordinal_logit_skin_tone, fdr_correct,
)

# H1: two-way ANOVA per system (interaction term)
anova_results = []
for sys_name, sub in synth_audit.groupby("auditor"):
    res = two_way_anova(sub.assign(error_f=sub["error"].astype(float)), dv="error_f",
                        factor_a="skin_tone", factor_b="gender")
    interaction = res[res["Source"] == "skin_tone * gender"]
    if len(interaction):
        anova_results.append({"system": sys_name, "p_interaction": float(interaction.iloc[0]["p-unc"])})
anova_df = pd.DataFrame(anova_results)
rejected, corrected = fdr_correct(anova_df["p_interaction"].tolist())
anova_df["p_fdr"] = corrected
anova_df["H1_supported"] = rejected
print("H1 (intersectional disparity):")
anova_df

In [ ]:
# H2: McNemar's flip test, skin tone 1 vs 6, per system
mcn_rows = []
for sys_name, sub in synth_audit.groupby("auditor"):
    res = mcnemars_paired(sub, pair_col="seed_identity_id", condition_col="skin_tone",
                          outcome_col="error", cond_a_value=1, cond_b_value=6)
    mcn_rows.append({"system": sys_name, **res})
mcn_df = pd.DataFrame(mcn_rows)
rejected, corrected = fdr_correct(mcn_df["pvalue"].tolist())
mcn_df["p_fdr"] = corrected
mcn_df["H2_supported"] = rejected
print("H2 (counterfactual flip rate):")
mcn_df

In [ ]:
# H3: ordinal logit on skin tone, per system
logit_rows = []
for sys_name, sub in synth_audit.groupby("auditor"):
    sub = sub.assign(skin_tone=sub["skin_tone"].astype(int),
                     gender_num=(sub["gender"] == "female").astype(int))
    model = ordinal_logit_skin_tone(sub, skin_tone_col="skin_tone", error_col="error",
                                    covariates=["gender_num"])
    logit_rows.append({
        "system": sys_name,
        "skin_tone_coef": float(model.params["skin_tone"]),
        "skin_tone_p": float(model.pvalues["skin_tone"]),
        "odds_ratio": float(np.exp(model.params["skin_tone"])),
    })
logit_df = pd.DataFrame(logit_rows)
print("H3 (continuous skin tone effect):")
logit_df

In [ ]:
# Intersectional error table (Gender Shades style) for one system
subset = synth_audit[synth_audit["auditor"] == "aws_rekognition"]
tbl = intersectional_error_table(subset)
tbl.pivot(index="skin_tone", columns="gender", values="error_rate").round(3)

## 5. Visualization

Render the publication-quality static figures and the interactive Plotly dashboard. If everything renders without error, our viz layer is ready to consume real audit results.

In [ ]:
import matplotlib.pyplot as plt

from cap.viz import (
    apply_paper_style, intersectional_error_heatmap, confidence_violin,
    skin_tone_regression_plot, system_comparison_radar, system_task_grid,
    save_figure_all_formats,
)

apply_paper_style(dpi=120)  # screen-friendly DPI for the notebook

# Build an intersectional table per system
fig_dir = REPO_ROOT / "notebooks" / "smoke_outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
for sys_name, sub in synth_audit.groupby("auditor"):
    tbl = intersectional_error_table(sub)
    fig = intersectional_error_heatmap(tbl, value_col="error_rate", title=f"Error rate — {sys_name}")
    save_figure_all_formats(fig, fig_dir / f"intersectional_{sys_name}")
    plt.show()
    plt.close(fig)

In [ ]:
# Confidence violin
fig = confidence_violin(synth_audit, group_col="skin_tone", hue_col="gender",
                        title="Confidence by demographic (synthetic)")
plt.show(); plt.close(fig)

# Continuous skin-tone regression (per-system error rate)
agg = (synth_audit.groupby(["auditor", "skin_tone"])
       .agg(error_rate=("error", "mean"))
       .reset_index())
fig = skin_tone_regression_plot(agg, system_col="auditor")
plt.show(); plt.close(fig)

In [ ]:
# Cross-system comparison: radar + grid
system_summary = (synth_audit.groupby("auditor")["error"].mean()
                  .rsub(1).rename("gender_accuracy").reset_index())
system_summary["detection_rate"] = 0.99  # synthetic stand-in
system_summary["confidence_avg"] = synth_audit.groupby("auditor")["confidence"].mean().values

fig = system_comparison_radar(system_summary, system_col="auditor",
                              metric_cols=["gender_accuracy", "detection_rate", "confidence_avg"])
plt.show(); plt.close(fig)

task_grid = pd.DataFrame({
    "auditor": np.repeat(systems, 2),
    "task": ["gender", "age"] * len(systems),
    "accuracy": rng.uniform(0.7, 0.97, size=2 * len(systems)),
})
fig = system_task_grid(task_grid)
plt.show(); plt.close(fig)

In [ ]:
# Interactive Plotly dashboard
from cap.viz import build_interactive_dashboard

agg_for_dashboard = synth_audit.copy()
agg_for_dashboard["error_rate"] = agg_for_dashboard["error"]
agg_for_dashboard["flip_rate"] = agg_for_dashboard.groupby(["auditor", "seed_identity_id"])[
    "prediction"
].transform("nunique").gt(1).astype(int)

dash_path = REPO_ROOT / "notebooks" / "smoke_outputs" / "dashboard.html"
build_interactive_dashboard(agg_for_dashboard, output_html=dash_path,
                            title="CAP Smoke Test Dashboard (synthetic data)")
print(f"Dashboard written: {dash_path}")
print("Open in a browser to inspect.")

## 6. Summary

If every cell above ran green:

- ✓ Imports + library wiring are correct
- ✓ Prompt construction holds invariants
- ✓ Stratified sampling distributes balance correctly
- ✓ ANOVA / McNemar's / ordinal logit / FDR all work on the audit schema
- ✓ Static figures render in publication formats (PDF/SVG/PNG)
- ✓ Interactive Plotly dashboard exports to HTML

**Next step:** GCP setup + first L4 smoke run on 5 real images.

**Outputs from this notebook are in `notebooks/smoke_outputs/`** — inspect figures and the dashboard HTML to confirm they look reasonable.

## 3b. (Optional) Real FairFace download

Skip unless you want to verify the live HuggingFace path. Pulls ~2 GB and creates 50 sample seeds in `data/fairface/`.

In [ ]:
# from cap.data import load_or_sample_seeds
# seeds = load_or_sample_seeds(
#     output_dir=REPO_ROOT / "data" / "fairface",
#     n=50,
#     stratify_by=["race", "gender", "age"],
#     seed=42,
# )
# print(f"Loaded {len(seeds)} real seeds. First: {seeds[0]}")